In [1]:
import json
from pathlib import Path
from collections import defaultdict
import re

In [6]:
def find_json_files(base_path):
    """Find all JSON files in the directory structure."""
    json_files = []
    base = Path(base_path)
    
    for item in base.rglob("*.json"):
        json_files.append(item)
    
    return json_files


def extract_letter_id_from_path(json_path):
    """Extract letter ID from the file path.
    
    Expected structure: .../annotations-json/12345.txt/admin.json
    Returns: 12345
    """
    # Get the parent directory name (e.g., "12345.txt")
    parent_dir = json_path.parent.name
    
    # Extract just the number part
    match = re.match(r'(\d+)\.txt', parent_dir)
    if match:
        return match.group(1)
    
    return None


def parse_inception_json(json_path):
    """Parse a single INCEpTION JSON file and extract annotations."""
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    # Extract letter ID from path
    letter_id = extract_letter_id_from_path(json_path)
    
    # Find the Sofa (text content)
    sofa_string = None
    for item in data.get('%FEATURE_STRUCTURES', []):
        if item.get('%TYPE') == 'uima.cas.Sofa':
            sofa_string = item.get('sofaString', '')
            break
    
    # Extract PatristicReference annotations
    annotations = []
    for item in data.get('%FEATURE_STRUCTURES', []):
        if item.get('%TYPE') == 'webanno.custom.PatristicReference':
            annotations.append({
                'annotation_id': item['%ID'],
                'begin': item['begin'],
                'end': item['end'],
                'church_fathers': item.get('church_fathers', ''),
                'confidence': item.get('confidence', ''),
                'detection_source': item.get('detection_source', ''),
                'patristic_source': item.get('patristic_source', ''),
                'patristic_text': item.get('patristic_text', ''),
                'reference_type': item.get('reference_type', ''),
                'sofa_string': sofa_string,
                'source_file': json_path.name,
                'letter_id': letter_id  # Add letter ID
            })
    
    return annotations


def extract_unique_patristics(all_annotations):
    """Extract unique patristic texts and assign canonical IDs."""
    patristic_texts = {}
    patristic_counter = 1
    
    for ann in all_annotations:
        # Use patristic_source as unique key
        key = ann['patristic_source']
        
        if key and key not in patristic_texts:
            # Extract author name (first part before comma)
            author = ann['patristic_source'].split(',')[0].strip() if ann['patristic_source'] else 'Unknown'
            
            patristic_texts[key] = {
                'patristic_id': f'PAT_{patristic_counter:03d}',
                'patristic_text': ann['patristic_text'],
                'source': ann['patristic_source'],
                'church_father_id': ann['church_fathers'],
                'church_father': author
            }
            patristic_counter += 1
    
    return patristic_texts


def create_queries(all_annotations, patristic_texts):
    """Create queries from annotations with ground truth links."""
    queries = []
    
    for ann in all_annotations:
        # Extract Bullinger passage text
        query_text = ann['sofa_string'][ann['begin']:ann['end']] if ann['sofa_string'] else ''
        
        # Find matching patristic_id
        key = ann['patristic_source']
        patristic_id = patristic_texts.get(key, {}).get('patristic_id', None)
        
        if patristic_id and query_text:  # Only include if we have both
            query = {
                'query_id': f"BULL_{ann['annotation_id']:04d}",
                'annotation_id': ann['annotation_id'],
                'letter_id': ann['letter_id'],  # Add letter ID
                'query_text': query_text,
                'ground_truth_patristic_ids': [patristic_id],
                'metadata': {
                    'reference_type': ann['reference_type'],
                    'confidence': ann['confidence'],
                    'detection_source': ann['detection_source'],
                    'source_file': ann['source_file']
                }
            }
            queries.append(query)
    
    return queries


def extract_unique_letters(queries):
    """Extract unique letter IDs from queries."""
    letter_ids = set()
    for query in queries:
        if query['letter_id']:
            letter_ids.add(query['letter_id'])
    
    return sorted(list(letter_ids))


def main():
    # Configuration
    base_path = "../annotations/annotations-json"
    output_dir = Path("../annotations/evaluation_dataset")
    output_dir.mkdir(exist_ok=True)
    
    print("Finding JSON files.")
    json_files = find_json_files(base_path)
    print(f"Found {len(json_files)} JSON files")
    
    print("Step 2: Parsing annotations")
    all_annotations = []
    for json_file in json_files:
        annotations = parse_inception_json(json_file)
        all_annotations.extend(annotations)
        letter_id = extract_letter_id_from_path(json_file)
        print(f"  {json_file.name} (Letter {letter_id}): {len(annotations)} annotations")
    
    print(f"Total annotations: {len(all_annotations)}")
    
    print(" Extracting unique patristic texts.")
    patristic_texts = extract_unique_patristics(all_annotations)
    print(f"Found {len(patristic_texts)} unique patristic texts")
    
    # Save candidates
    candidates_file = output_dir / "candidates.jsonl"
    with open(candidates_file, 'w', encoding='utf-8') as f:
        for candidate in patristic_texts.values():
            f.write(json.dumps(candidate, ensure_ascii=False) + '\n')
    print(f"\nSaved candidates to {candidates_file}")
    
    print("Creating queries.")
    queries = create_queries(all_annotations, patristic_texts)
    print(f"Created {len(queries)} queries")
    
    # Save queries
    queries_file = output_dir / "queries.jsonl"
    with open(queries_file, 'w', encoding='utf-8') as f:
        for query in queries:
            f.write(json.dumps(query, ensure_ascii=False) + '\n')
    print(f"Saved queries to {queries_file}")
    
    # Extract unique letters
    unique_letters = extract_unique_letters(queries)
    print(f"\nFound {len(unique_letters)} unique letters")
    
    # Save letter list
    letters_file = output_dir / "letters.json"
    with open(letters_file, 'w', encoding='utf-8') as f:
        json.dump({'letter_ids': unique_letters}, f, indent=2)
    print(f"Saved letter list to {letters_file}")

    # Extract sources 
    sources_only = [
        {
            "patristic_id": entry["patristic_id"],
            "source": entry["source"]
        }
        for entry in patristic_texts.values()
    ]

    # Save source list
    sources_file = output_dir / "sources.json"
    with open(sources_file, 'w', encoding='utf-8') as f:
        json.dump({'patristic_source': sources_only}, f, indent=2)
    print(f"Saved source list to {sources_file}")
    
    print("Dataset creation complete!")


if __name__ == "__main__":
    main()

Finding JSON files.
Found 67 JSON files
Step 2: Parsing annotations
  CURATION_USER3640810787533886868.json (Letter 12805): 1 annotations
  CURATION_USER8704988096063683106.json (Letter 12408): 1 annotations
  CURATION_USER7166710723600358648.json (Letter 9183): 1 annotations
  CURATION_USER5280774515238020227.json (Letter 11507): 2 annotations
  CURATION_USER10584918441564715329.json (Letter 9780): 1 annotations
  CURATION_USER14995620711282683404.json (Letter 3904): 1 annotations
  CURATION_USER2573837776675940556.json (Letter 13088): 5 annotations
  CURATION_USER17890577175362855848.json (Letter 10768): 1 annotations
  CURATION_USER9781862005225563710.json (Letter 11689): 3 annotations
  CURATION_USER16391865541975423727.json (Letter 1888): 1 annotations
  CURATION_USER7710996535731708925.json (Letter 3425): 1 annotations
  CURATION_USER14723529173670205661.json (Letter 12046): 1 annotations
  CURATION_USER3479073230119956466.json (Letter 10484): 2 annotations
  CURATION_USER7801182